In [1]:
from transformers import MarianMTModel, MarianTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import load_dataset
import torch
from torch.utils.data import DataLoader
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
import os

In [2]:
file_path = "../files/kilba_english.json"
dataset = load_dataset("json", data_files={"train": file_path})
train_dataset = dataset["train"]

In [ ]:
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name, use_auth_token=os.getenv("HUGGING_FACE_TOKEN"))
model = MarianMTModel.from_pretrained(model_name, use_auth_token=os.getenv("HUGGING_FACE_TOKEN"))

In [ ]:
def preprocess_data(batch):
    inputs = tokenizer(batch["kilba"], max_length=128, truncation=True, padding="max_length")
    targets = tokenizer(batch["english"], max_length=128, truncation=True, padding="max_length")
    batch["input_ids"] = inputs["input_ids"]
    batch["attention_mask"] = inputs["attention_mask"]
    batch["labels"] = targets["input_ids"]
    return batch

In [ ]:
tokenized_dataset = train_dataset.map(preprocess_data, batched=True)

In [ ]:
def collate_fn(batch):
    input_ids = torch.tensor([item["input_ids"] for item in batch])
    attention_mask = torch.tensor([item["attention_mask"] for item in batch])
    labels = torch.tensor([item["labels"] for item in batch])
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [ ]:
train_loader = DataLoader(tokenized_dataset, batch_size=8, collate_fn=collate_fn)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",  # Save directory
    evaluation_strategy="epoch",  # Evaluate after every epoch
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    save_total_limit=2,  # Save only the last 2 models
    predict_with_generate=True  # Use model.generate() for predictions
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

In [ ]:
def translate_sentence(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", max_length=128, truncation=True)
    outputs = model.generate(inputs["input_ids"], max_length=128, num_beams=5, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Test translation
translated_text = translate_sentence("1Nja yah Yesu avu Baitalami atə hə'i Yahudiya, aku bəji nda təl Hiridus ku dzəga təlkur ti.")
print("Translated Text:", translated_text)

In [ ]:
# Test references and hypothesis
references = [["Now when Jesus was born in Bethlehem of Judaea in the days of Herod the king."]]
hypothesis = ["Now when Jesus was born in Bethlehem of Judaea in the days of Herod the king"]

bleu_score = sentence_bleu(references, hypothesis)
print("BLEU Score:", bleu_score)

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
scores = scorer.score("Now when Jesus was born in Bethlehem of Judaea in the days of Herod the king.",
                      "Now when Jesus was born in Bethlehem of Judaea in the days of Herod the king")
print(scores)